# DocuXray — End-to-End Grounding Pipeline (Colab)

This notebook runs the DocuXray document field grounding pipeline in order:

1. Upload the document image
2. Upload the annotation / field JSON (the fields you want grounded)
3. Generate OCR JSON (PaddleOCR)
4. Run document graph generation (image + OCR JSON)
5. Run the grounding pipeline (image + fields JSON + OCR JSON)
6. Draw grounded boxes on the image

Run the cells top to bottom. Each step's outputs feed directly into the next step.


## Step 0 — Install dependencies

PaddleOCR/PaddlePaddle for OCR, OpenCV for drawing, Pillow for image I/O.

In [110]:
!pip install -q paddlepaddle paddleocr opencv-python-headless pillow

## Step 0 (continued) — Upload and set up the DocuXray package

Upload `docuxray_colab.zip` (the DocuXray source package). It will be unzipped to
`/content/docuxray_colab`, and its `src/` directory (containing the importable
`docuxray` package) is added to `sys.path`. No `pip install` is required —
the package only needs `Pillow`, which Colab already has.

In [ ]:
from google.colab import files

print("Please select docuxray_colab.zip")
docuxray_zip = files.upload()

Please select docuxray_colab.zip


In [ ]:
import zipfile

zip_name = list(docuxray_zip.keys())[0]

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall("/content")

print(f"Extracted {zip_name} to /content")

In [ ]:
import sys

DOCUXRAY_ROOT = "/content/docuxray_colab"

for path in (DOCUXRAY_ROOT, f"{DOCUXRAY_ROOT}/src"):
    if path not in sys.path:
        sys.path.insert(0, path)

import docuxray  # sanity check that the package is importable

print("docuxray package is importable from:", docuxray.__file__)

## Step 1 — Upload the document image

In [ ]:
from google.colab import files
from IPython.display import Image, display

print("Please select the document image (e.g. an invoice .png/.jpg)")
uploaded_image = files.upload()

image_path = list(uploaded_image.keys())[0]
print("Uploaded image:", image_path)

display(Image(filename=image_path))

## Step 2 — Upload the annotation / field JSON

This is the requested-fields JSON that DocuXray will try to ground, in the
`{"fields": [...]}` format (also accepts `{"records": [...]}` or a flat
`{"key": "value"}` object — see the DocuXray README for details).

In [ ]:
print("Please select the annotation / field JSON")
uploaded_fields = files.upload()

fields_json_path = list(uploaded_fields.keys())[0]
print("Uploaded fields JSON:", fields_json_path)

import json

with open(fields_json_path, "r", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2)[:2000])


## Step 3 — Generate OCR JSON

This runs PaddleOCR on the uploaded image and writes a DocuXray-compatible OCR
JSON (`{"words": [{"id", "text", "box", "confidence", "page"}, ...]}`).

This uses PaddleOCR's modern `predict()` API (PaddleOCR 3.x), not the legacy
`ocr()` API that DocuXray's bundled OCR adapter expects — so we generate the
OCR JSON here ourselves and pass it as `ocr_json_path=` in the later steps,
rather than letting DocuXray call PaddleOCR internally.

In [ ]:
import json
from pathlib import Path

from paddleocr import PaddleOCR


def polygon_to_box(points):
    xs = [float(point[0]) for point in points]
    ys = [float(point[1]) for point in points]
    return [
        round(min(xs)),
        round(min(ys)),
        round(max(xs)),
        round(max(ys)),
    ]


def generate_ocr_json(image_path, output_path=None, lang="en", min_confidence=0.0):
    """Run PaddleOCR and write a DocuXray-compatible OCR JSON file."""
    image_path = Path(image_path)

    if output_path is None:
        output_path = f"{image_path.stem}.ocr.json"

    ocr = PaddleOCR(
        lang=lang,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        enable_mkldnn=False,  # works around a oneDNN/PIR bug in PaddlePaddle 3.3.x
        # (NotImplementedError: ConvertPirAttribute2RuntimeAttribute ...)
    )

    results = ocr.predict(str(image_path))

    words = []
    for page in results:
        rec_texts = page.get("rec_texts", [])
        rec_scores = page.get("rec_scores", [])
        rec_polys = page.get("rec_polys", [])

        for text, score, poly in zip(rec_texts, rec_scores, rec_polys):
            text = str(text).strip()
            if not text or score < min_confidence:
                continue
            words.append(
                {
                    "id": f"w{len(words)}",
                    "text": text,
                    "box": polygon_to_box(poly),
                    "confidence": float(score),
                    "page": 0,
                }
            )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump({"words": words}, f, indent=2, ensure_ascii=False)

    return str(output_path)


In [ ]:
ocr_json_path = generate_ocr_json(image_path, output_path="ocr_output.json")

print("OCR JSON saved to:", ocr_json_path)

with open(ocr_json_path, "r", encoding="utf-8") as f:
    ocr_data = json.load(f)

print(f"Detected {len(ocr_data['words'])} words")
print(json.dumps(ocr_data["words"][:5], indent=2))


## Step 4 — Run document graph generation

Builds the DocuXray document graph (pages, OCR words, lines, and any table
structure) from the image and the OCR JSON generated in Step 3, and writes it
to `document_graph.json`.

In [ ]:
from generate_document_graph import generate_document_graph

document_graph_path = generate_document_graph(
    image_path=image_path,
    ocr_json_path=ocr_json_path,
    output_path="document_graph.json",
)

print("Document graph written to:", document_graph_path)

with open(document_graph_path, "r", encoding="utf-8") as f:
    document_graph = json.load(f)

print("Graph top-level keys:", list(document_graph.keys()))


## Step 5 — Run the grounding pipeline

Runs DocuXray's full grounding pipeline using the image, the field requests
JSON from Step 2, and the OCR JSON from Step 3. For each requested field this
returns the matched text, bounding box, confidence, and status
(`matched`, `low_confidence`, `ambiguous`, or `not_found`).

In [ ]:
from docuxray.pipeline import run_grounding_pipeline

grounded_fields = run_grounding_pipeline(
    image_path=image_path,
    fields_path=fields_json_path,
    ocr_json_path=ocr_json_path,
    output_graph_path="document_graph_grounded.json",
)

grounded_results = [field.to_dict() for field in grounded_fields]

with open("grounding_results.json", "w", encoding="utf-8") as f:
    json.dump(grounded_results, f, indent=2, ensure_ascii=False)

print(f"Grounded {len(grounded_results)} field(s)")
print(json.dumps(grounded_results, indent=2))


## Step 6 — Draw grounded boxes

Draws the bounding box for each grounded field onto the document image.
Box color reflects the grounding status:

- **green** — `matched`
- **orange** — `low_confidence`
- **yellow** — `ambiguous`
- fields with status `not_found` have no box and are listed separately below.

In [ ]:
import cv2

STATUS_COLORS = {
    "matched": (0, 200, 0),        # green (BGR)
    "low_confidence": (0, 165, 255),  # orange
    "ambiguous": (0, 255, 255),    # yellow
}

image = cv2.imread(image_path)
if image is None:
    raise FileNotFoundError(f"Could not read image: {image_path}")

not_found = []

for result in grounded_results:
    box = result.get("bbox")
    if not box:
        not_found.append(result)
        continue

    x1, y1, x2, y2 = box
    color = STATUS_COLORS.get(result["status"], (255, 0, 0))

    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)

    label = f"{result['key']}={result['value']}"
    label_y = max(y1 - 8, 12)
    cv2.putText(
        image,
        label,
        (x1, label_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        color,
        1,
        cv2.LINE_AA,
    )

output_image_path = "grounded_boxes.jpg"
cv2.imwrite(output_image_path, image)

print(f"Drew {len(grounded_results) - len(not_found)} box(es)")
print(f"Saved annotated image to: {output_image_path}")

if not_found:
    print("\nFields with no grounded box (status=not_found):")
    for result in not_found:
        print(f"  - {result['key']} = {result['value']} (field_id={result['field_id']})")


In [ ]:
from IPython.display import Image as IPImage, display

display(IPImage(filename=output_image_path))

In [ ]:
from google.colab import files

files.download(output_image_path)
files.download("grounding_results.json")
